# <img align="left" src="./images/movie_camera.png"     style=" width:40px;  " > 实践实验室：协同过滤推荐系统

在本练习中，你将实现协同过滤，以构建一个电影推荐系统。

# <img align="left" src="./images/film_reel.png"     style=" width:40px;  " > 大纲
- [1 - 符号表示](#1)
- [2 - 推荐系统](#2)
- [3 - 电影评分数据集](#3)
- [4 - 协同过滤学习算法](#4)
  - [4.1 协同过滤成本函数](#4.1)
    - [练习1](#ex01)
- [5 - 学习电影推荐](#5)
- [6 - 推荐内容](#6)
- [7 - 恭喜！](#7)




##  软件包 <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;   " >
我们将使用现在已经熟悉的NumPy和TensorFlow软件包。

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from recsys_utils import *

<a name="1"></a>
## 1 - 符号表示


|通用 <br />  符号表示  | 描述| Python (if any) |
|:-------------|:------------------------------------------------------------|:-------|
| $r(i,j)$     | 标量；若用户j对电影i进行了评分，则为1；否则为0             ||
| $y(i,j)$     | 标量；若$r(i,j)=1$，则为用户j对电影i的评分  ||
|$\mathbf{w}^{(j)}$|向量；用户j的参数||
|$b^{(j)}$|标量；用户j的参数||
|$\mathbf{x}^{(i)}$|向量；电影i的特征评分||
|$n_u$|用户数量|num_users|
|$n_m$|电影数量|num_movies|
|$n$|特征数量|num_features|
|$\mathbf{X}$|向量$\mathbf{x}^{(i)}$构成的矩阵|X|
|$\mathbf{W}$|向量$\mathbf{w}^{(j)}$构成的矩阵|W|
|$\mathbf{b}$|偏差参数$b^{(j)}$构成的向量|b|
|$\mathbf{R}$|元素$r(i, j)$构成的矩阵|R|
|$\mathbf{m}^{(j)}$|用户j给几个电影打了分||
|$\mathbf{w^{(j)} \cdot x^{(i)} + b^{(j)}}$|预测用户j对电影i的评分||



<a name="2"></a>
## 2 - 推荐系统 <img align="left" src="./images/film_rating.png" style=" width:40px;  " >
在本实验中，你将实现协同过滤学习算法，并将其应用于电影评分数据集。

协同过滤推荐系统的目标是生成两个向量：对于每个用户，生成一个“参数向量”，体现该用户的电影喜好。对于每部电影，生成一个大小相同的特征向量，体现该电影的某些描述。这两个向量的点积加上偏差项应得出对该用户可能给该电影的评分的估计值。

下面的图表详细说明了这些向量是如何学习得到的。

<figure>
   <img src="./images/ColabFilterLearn.PNG"  style="width:740px;height:250px;" >
</figure>

现有评分以矩阵形式提供，如下所示。$Y$ 包含评分；评分范围从 0.5 到 5（含），步长为 0.5。如果电影未被评分，则为 0。$R$ 在电影已被评分的位置处为 1。电影在行，用户在列。每个用户都有一个参数向量 $w^{user}$ 和偏差。每部电影都有一个特征向量 $x^{movie}$。利用现有的用户/电影评分作为训练数据，同时学习这些向量。上面展示了一个训练示例：$\mathbf{w}^{(1)} \cdot \mathbf{x}^{(1)} + b^{(1)} = 4$。值得注意的是，特征向量 $x^{movie}$ 必须满足所有用户，而用户向量 $w^{user}$ 必须满足所有电影。这就是这种方法名称的由来——所有用户协作生成评分集。 

<figure>
   <img src="./images/ColabFilterUse.PNG"  style="width:640px;height:250px;" >
</figure>

一旦学习到特征向量和参数，就可以用它们来预测用户对未评分电影的评分。这在上面的图表中有所展示。该公式是预测用户一对电影零的评分的一个示例。

在本练习中，你将实现函数`cofiCostFunc`，该函数用于计算协同过滤目标函数。在实现目标函数之后，你将使用TensorFlow自定义训练循环来学习协同过滤的参数。第一步是详细介绍本实验中将会用到的数据集和数据结构。

<a name="3"></a>
## 3 - 电影评分数据集 <img align="left" src="./images/film_rating.png"     style=" width:40px;  " >
该数据集源自[MovieLens “ml - latest - small”](https://grouplens.org/datasets/movielens/latest/)数据集。  
[F. 麦克斯韦·哈珀和约瑟夫·A. 康斯坦。2015年。《MovieLens数据集：历史与背景》。《ACM交互式智能系统汇刊》（TiiS）5，4：19：1 - 19：19。<https://doi.org/10.1145/2827872>]

原始数据集包含600名用户对9000部电影的评分。为了聚焦于2000年以来的电影，数据集规模有所缩减。该数据集的评分范围为0.5到5，以0.5为步长递增。缩减后的数据集有$n_u = 443$名用户和$n_m = 4778$部电影。

接下来，你将把电影数据集加载到变量$Y$和$R$中。
矩阵$Y$（一个$n_m \times n_u$矩阵）存储评分$y^{(i,j)}$。矩阵$R$是一个二进制指示矩阵，若用户$j$对电影$i$进行了评分，则$R(i,j) = 1$，否则$R(i,j) = 0$。

在本部分练习中，你还将使用矩阵$\mathbf{X}$、$\mathbf{W}$和$\mathbf{b}$： 

$$\mathbf{X} = 
\begin{bmatrix}
--- (\mathbf{x}^{(0)})^T --- \\
--- (\mathbf{x}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{x}^{(n_m-1)})^T --- \\
\end{bmatrix} , \quad
\mathbf{W} = 
\begin{bmatrix}
--- (\mathbf{w}^{(0)})^T --- \\
--- (\mathbf{w}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{w}^{(n_u-1)})^T --- \\
\end{bmatrix},\quad
\mathbf{ b} = 
\begin{bmatrix}
 b^{(0)}  \\
 b^{(1)} \\
\vdots \\
b^{(n_u-1)} \\
\end{bmatrix}\quad
$$ 

$\mathbf{X}$的第$i$行对应第$i$部电影的特征向量$x^{(i)}$，$\mathbf{W}$的第$j$行对应第$j$个用户的一个参数向量$\mathbf{w}^{(j)}$ 。$x^{(i)}$和$\mathbf{w}^{(j)}$都是$n$维向量。在本练习中，你将使用$n = 10$，因此，$\mathbf{x}^{(i)}$和$\mathbf{w}^{(j)}$有10个元素。相应地，$\mathbf{X}$是一个$n_m \times 10$矩阵，$\mathbf{W}$是一个$n_u \times 10$矩阵。

我们将从加载电影评分数据集开始，以了解数据的结构。我们将使用电影数据集加载$Y$和$R$。我们还将加载具有预先计算值的$\mathbf{X}$、$\mathbf{W}$和$\mathbf{b}$。这些值将在本实验的后续部分学习，但我们将使用预先计算的值来开发成本模型。 

In [2]:
#Load data
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y, R = load_ratings_small()

print("Y", Y.shape, "R", R.shape)
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("num_features", num_features)
print("num_movies",   num_movies)
print("num_users",    num_users)

Y (4778, 443) R (4778, 443)
X (4778, 10)
W (443, 10)
b (1, 443)
num_features 10
num_movies 4778
num_users 443


In [3]:
#  From the matrix, we can compute statistics like average rating.
tsmean =  np.mean(Y[0, R[0, :].astype(bool)])
print(f"Average rating for movie 1 : {tsmean:0.3f} / 5" )

Average rating for movie 1 : 3.400 / 5


<a name="4"></a>
## 4 - 协同过滤学习算法 <img align="left" src="./images/film_filter.png"     style=" width:40px;  " >

现在，你将开始实现协同过滤学习算法。你将从实现目标函数入手。在电影推荐场景下的协同过滤算法考虑一组$n$维参数向量$\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m - 1)}$、$\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u - 1)}$以及$b^{(0)},...,b^{(n_u - 1)}$，其中模型对用户$j$对电影$i$的评分预测为$y^{(i,j)} = \mathbf{w}^{(j)}\cdot \mathbf{x}^{(i)} + b^{(i)}$ 。给定一个由一些用户对一些电影给出的评分数据集，你希望学习到能产生最佳拟合（最小化均方误差）的参数向量$\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m - 1)}$、$\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u - 1)}$以及$b^{(0)},...,b^{(n_u - 1)}$。

你将在`cofiCostFunc`中完成代码，以计算协同过滤的成本函数。 


<a name="4.1"></a>
### 4.1 协同过滤成本函数   

协同过滤成本函数由下式给出
$$J({\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},b^{(0)},...,\mathbf{w}^{(n_u-1)},b^{(n_u-1)}})= \frac{1}{2}\sum_{(i,j):r(i,j)=1}(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2
+\underbrace{
\frac{\lambda}{2}
\sum_{j=0}^{n_u-1}\sum_{k=0}^{n-1}(\mathbf{w}^{(j)}_k)^2
+ \frac{\lambda}{2}\sum_{i=0}^{n_m-1}\sum_{k=0}^{n-1}(\mathbf{x}_k^{(i)})^2
}_{regularization}
\tag{1}$$
（1）中的第一个求和是“对于所有满足$r(i,j)$等于1的$i$、$j$”，可以写成：
$$
= \frac{1}{2}\sum_{j=0}^{n_u-1} \sum_{i=0}^{n_m-1}r(i,j)*(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2
+\text{regularization}
$$

你现在应该编写协同过滤成本函数`cofiCostFunc`，以返回这个成本。

<a name="ex01"></a>
### 练习1
**for循环实现**：  
首先使用for循环实现成本函数。  
考虑分两步开发成本函数。第一步，开发无正则化的成本函数。下面提供了一个不包含正则化的测试用例来测试你的实现。一旦这一步可行，添加正则化并运行包含正则化的测试。注意，只有当$R(i,j) = 1$时，才应累积用户$j$和电影$i$的成本。 

In [4]:
# GRADED FUNCTION: cofi_cost_func
# UNQ_C1

def cofi_cost_func(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    nm, nu = Y.shape
    J = 0
    ### START CODE HERE ### 
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += np.square(np.dot(w,x) + b_j - y) * r
    J += lambda_* (np.sum(np.square(W)) + np.sum(np.square(X)))
    J = J/2
    ### END CODE HERE ### 

    return J

In [5]:
# Public tests
from public_tests import *
test_cofi_cost_func(cofi_cost_func)

All tests passed!


<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
    You can structure the code in two for loops similar to the summation in (1).   
    Implement the code without regularization first.   
    Note that some of the elements in (1) are vectors. Use np.dot(). You can also use np.square().
    Pay close attention to which elements are indexed by i and which are indexed by j. Don't forget to divide by two.
    
```python     
    ### START CODE HERE ###  
    for j in range(nu):
        
        
        for i in range(nm):
            
            
    ### END CODE HERE ### 
```    
<details>
    <summary><font size="2" color="darkblue"><b> Click for more hints</b></font></summary>
        
    Here is some more details. The code below pulls out each element from the matrix before using it. 
    One could also reference the matrix directly.  
    This code does not contain regularization.
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = 
            y = 
            r =
            J += 
    J = J/2
    ### END CODE HERE ### 

```
    
<details>
    <summary><font size="2" color="darkblue"><b>Last Resort (full non-regularized implementation)</b></font></summary>
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += np.square(r * (np.dot(w,x) + b_j - y ) )
    J = J/2
    ### END CODE HERE ### 
```
    
<details>
    <summary><font size="2" color="darkblue"><b>regularization</b></font></summary>
     Regularization just squares each element of the W array and X array and them sums all the squared elements.
     You can utilize np.square() and np.sum().

<details>
    <summary><font size="2" color="darkblue"><b>regularization details</b></font></summary>
    
```python 
    J += lambda_* (np.sum(np.square(W)) + np.sum(np.square(X)))
```
    
</details>
</details>
</details>
</details>

    


In [6]:
# 缩小数据集规模，以便程序运行得更快
num_users_r = 4
num_movies_r = 5 
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
b_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]

# 评估代价函数
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

Cost: 13.67


**Expected Output (lambda = 0)**:  
$13.67$.

In [7]:
# 评估带正则化的代价函数
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost (with regularization): 28.09


**Expected Output**:

28.09

**向量化实现**
创建一个向量化实现来计算 $J$ 非常重要，因为在优化过程中计算成本的函数会被多次调用。这里所使用的线性代数并非本系列的重点，因此提供了具体实现。如果你是线性代数方面的专家，可以在不参考以下代码的情况下创建自己的版本。   

运行以下代码，并验证其产生的结果与非向量化版本相同。

In [8]:
def cofi_cost_func_v(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Vectorized for speed. Uses tensorflow operations to be compatible with custom training loop.
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y)*R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
    return J

In [9]:
# Evaluate cost function
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

# Evaluate cost function with regularization 
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost: 13.67
Cost (with regularization): 28.09


**Expected Output**:  
Cost: 13.67  
Cost (with regularization): 28.09

<a name="5"></a>
## 5 - 学习电影推荐<img align="left" src="./images/film_man_action.png" style=" width:40px;  " >


在完成协同过滤成本函数的实现后，你就可以开始训练算法，为自己生成电影推荐。   

在下面的单元格中，你可以输入自己的电影选择。然后算法将为你提供推荐！我们已根据自己的偏好填写了一些值，但在你使用我们的选择让程序正常运行后，你应该根据自己的口味进行更改。   

数据集中所有电影的列表位于文件[电影列表](data/small_movie_list.csv)中。

In [10]:
movieList, movieList_df = load_Movie_List_pd()

my_ratings = np.zeros(num_movies)          #  初始化我的评分

# 查看文件small_movie_list.csv，获取数据集中每部电影的ID
# 例如，《玩具总动员3》（2010年）的ID为2700，所以要给它评“5”分，你可以设置
my_ratings[2700] = 5 

# 或者假设你不喜欢《劝导》（2007年），你可以设置
my_ratings[2609] = 2;

# 我们挑选了一些喜欢或不喜欢的电影，给出的评分如下：
my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)
my_rated = [i for i in range(len(my_ratings)) if my_ratings[i] > 0]

print('\n我的评分:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');


我的评分:

Rated 5.0 for  Shrek (2001)
Rated 5.0 for  Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Rated 2.0 for  Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Rated 5.0 for  Harry Potter and the Chamber of Secrets (2002)
Rated 5.0 for  Pirates of the Caribbean: The Curse of the Black Pearl (2003)
Rated 5.0 for  Lord of the Rings: The Return of the King, The (2003)
Rated 3.0 for  Eternal Sunshine of the Spotless Mind (2004)
Rated 5.0 for  Incredibles, The (2004)
Rated 2.0 for  Persuasion (2007)
Rated 5.0 for  Toy Story 3 (2010)
Rated 3.0 for  Inception (2010)
Rated 1.0 for  Louis Theroux: Law & Disorder (2008)
Rated 1.0 for  Nothing to Declare (Rien à déclarer) (2010)


现在，让我们将这些评分添加到$Y$和$R$中，并对评分进行归一化处理。

In [11]:
# 重新加载评分并添加新评分
Y, R = load_ratings_small()
Y    = np.c_[my_ratings, Y]
R    = np.c_[(my_ratings != 0).astype(int), R]

# 归一化数据集
Ynorm, Ymean = normalizeRatings(Y, R)

让我们准备训练模型。初始化参数并选择Adam优化器。

In [12]:
#  Useful Values
num_movies, num_users = Y.shape
num_features = 100

# 设置初始参数（W，X），使用tf.Variable来跟踪这些变量
tf.random.set_seed(1234) # 为了得到一致的结果
W = tf.Variable(tf.random.normal((num_users,  num_features),dtype=tf.float64),  name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features),dtype=tf.float64),  name='X')
b = tf.Variable(tf.random.normal((1,          num_users),   dtype=tf.float64),  name='b')

# 实例化一个优化器。
optimizer = keras.optimizers.Adam(learning_rate=4e-1)

现在我们来训练协同过滤模型。该模型将学习参数$\mathbf{X}$、$\mathbf{W}$和$\mathbf{b}$ 。

同时学习$w$、$b$和$x$所涉及的操作并不属于TensorFlow神经网络包中提供的典型“层”。因此，课程2中使用的流程：Model、Compile()、Fit()、Predict()并不直接适用。相反，我们可以使用自定义训练循环。

回顾一下之前实验中梯度下降的步骤。
- 重复直至收敛：
    - 计算前向传播
    - 计算损失相对于参数的导数
    - 使用学习率和计算出的导数更新参数

TensorFlow具有为你计算导数的强大功能。如下所示。在`tf.GradientTape()`部分内，对TensorFlow变量的操作会被“录制”。当稍后调用`tape.gradient()`时，它将返回损失相对于被跟踪变量的梯度。然后可以使用优化器将这些梯度应用于参数。

这只是对TensorFlow和其他机器学习框架一个有用特性的非常简要的介绍。通过研究“自定义训练循环”，可以找到更多信息。 
    


In [13]:
iterations = 200
lambda_ = 1
for iter in range(iterations):
    # 使用TensorFlow的GradientTape
    # 记录用于计算成本的操作
    with tf.GradientTape() as tape:

        # 计算成本（成本中包含前向传播）
        cost_value = cofi_cost_func_v(X, W, b, Ynorm, R, lambda_)

    # 使用梯度带自动检索
    # 可训练变量相对于损失的梯度
    grads = tape.gradient( cost_value, [X,W,b] )

    # 通过更新变量值来执行一步梯度下降，以最小化损失
    optimizer.apply_gradients( zip(grads, [X,W,b]) )

    # 定期记录日志
    if iter % 20 == 0:
        print(f"Training loss at iteration {iter}: {cost_value:0.1f}")

Training loss at iteration 0: 2321191.3
Training loss at iteration 20: 122520.5
Training loss at iteration 40: 22811.3
Training loss at iteration 60: 5529.5
Training loss at iteration 80: 2579.5
Training loss at iteration 100: 2029.1
Training loss at iteration 120: 1897.3
Training loss at iteration 140: 1849.7
Training loss at iteration 160: 1825.3
Training loss at iteration 180: 1809.9


<a name="6"></a>
## 6 - 给一个用户推荐电影
下面，我们计算所有电影和用户的评分，并展示推荐的电影。这些推荐基于上面作为`my_ratings[]`输入的电影和评分。要预测用户 $j$ 对电影 $i$ 的评分，你需要计算 $\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)}$。这可以通过矩阵乘法对所有评分进行计算。

In [14]:
# 使用训练好的权重和偏差进行预测
p = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy()

# 恢复均值
pm = p + Ymean

my_predictions = pm[:,0]

# 对预测结果进行排序
ix = tf.argsort(my_predictions, direction='DESCENDING')

print("给我推荐的电影（我没评过分）：\n")
for i in range(17):
    j = ix[i]
    if j not in my_rated:
        print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\n原始评分与预测评分：\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

给我推荐的电影（我没评过分）：

Predicting rating 4.73 for movie Colourful (Karafuru) (2010)
Predicting rating 4.55 for movie Laggies (2014)
Predicting rating 4.55 for movie One I Love, The (2014)
Predicting rating 4.55 for movie Delirium (2014)
Predicting rating 4.55 for movie Kung Fu Panda: Secrets of the Masters (2011)
Predicting rating 4.55 for movie Battle Royale 2: Requiem (Batoru rowaiaru II: Chinkonka) (2003)
Predicting rating 4.55 for movie Into the Abyss (2011)
Predicting rating 4.55 for movie Eichmann (2007)
Predicting rating 4.54 for movie Particle Fever (2013)
Predicting rating 4.53 for movie Bossa Nova (2000)


原始评分与预测评分：

Original 5.0, Predicted 4.89 for Shrek (2001)
Original 5.0, Predicted 4.82 for Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Original 2.0, Predicted 2.18 for Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Original 5.0, Predicted 4.88 for Harry Potter and the Chamber of Secrets (2002)
Original 5.0, Predicted 4.86 f

在实践中，可以利用额外信息来改进我们的预测。前几百部电影的预测评分范围较小，我们可以从那些排名靠前的电影中，挑选出平均评分较高且评分数量超过20条的电影，以此来扩充上述内容。本节使用了一个[Pandas](https://pandas.pydata.org/)的data frame，它具有许多便捷的排序功能。

In [20]:
movieList_df["pred"] = my_predictions
movieList_df["my rating"] = my_ratings
filter=(movieList_df["number of ratings"] > 20)&(movieList_df["my rating"]<0.5)
movieList_df = movieList_df.reindex(columns=["pred", "my rating", "mean rating", "number of ratings", "title"])
movieList_df.loc[ix[:300]].loc[filter].sort_values("mean rating", ascending=False)

,pred,my rating,mean rating,number of ratings,title
2112,4.088934,0.0,4.238255,149,"Dark Knight, The (2008)"
155,4.182064,0.0,4.155914,93,Snatch (2000)
580,4.075562,0.0,4.155172,87,Spirited Away (Sen to Chihiro no kamikakushi) ...
393,4.331385,0.0,4.106061,198,"Lord of the Rings: The Fellowship of the Ring,..."
653,4.356261,0.0,4.021277,188,"Lord of the Rings: The Two Towers, The (2002)"
2804,4.073887,0.0,3.989362,47,Harry Potter and the Deathly Hallows: Part 1 (...
773,4.228898,0.0,3.960993,141,Finding Nemo (2003)
2649,4.072192,0.0,3.943396,53,How to Train Your Dragon (2010)
1051,4.029191,0.0,3.913978,93,Harry Potter and the Prisoner of Azkaban (2004)
361,4.083755,0.0,3.871212,132,"Monsters, Inc. (2001)"


先通过`movieList_df.loc[ix[:300]]`筛选出预估前300的电影      
然后通过`loc[filter]`筛选出评分数大于20并且我没看过的电影 

<a name="7"></a>
## 7 - 恭喜! <img align="left" src="./images/film_award.png"     style=" width:40px;  " >
你已经实现了一个有用的推荐系统！